# Run 1307 AI code from GCS in Colab

This notebook is meant to be opened from GitHub in Google Colab.
It syncs your `1307` code from a Google Cloud Storage bucket into `/content/1307`,
installs dependencies, checks GPU availability, and runs your entry script.

### Your Windows PC: `gcloud` and `gsutil` (optional)

Use this when you run **`gsutil`** from **PowerShell** on your laptop (for example syncing data to your bucket outside of Colab).

1. Install the [Google Cloud CLI](https://cloud.google.com/sdk/docs/install) — it includes **both** `gcloud` and `gsutil`.
2. **Restart Cursor** or open a **new** terminal so **PATH** includes `...\google-cloud-sdk\bin` (otherwise `gcloud` may be “not recognized”).
3. Confirm tools work:
   - `gcloud --version`
   - `gsutil version`
4. When you use this project on a machine for the first time (or after a long break):
   - `gcloud auth login`
   - `gcloud config set project YOUR_PROJECT_ID`  
   Use the **Project ID** from [Google Cloud Console](https://console.cloud.google.com/) (lowercase id), not the display name.

**Colab** uses the notebook’s in-runtime Google sign-in — that is **separate** from your PC. Keep **`GCP_PROJECT`** in the config cell below aligned with the same project.

## Progress tracker (edit as you go)

- [ ] Section 2 sync from GCS completed
- [ ] Section 4 dependencies installed
- [ ] Section 5 GPU confirmed (`nvidia-smi` works)
- [ ] Section 6 run directory created
- [ ] Section 7 dataset built or dataset path resolved
- [ ] Section 8 training run completed
- [ ] Section 9 outputs synced to GCS

### Next steps
- Set or verify `DOE_GRI_INPUT`.
- Build dataset (or set `DOE_DATASET_PATH` to an existing dataset directory).
- Run training.
- Sync run outputs + dataset artifacts to GCS.

## 1) Configure

Fill in these values before running the rest of the notebook.

In [ ]:
GCP_PROJECT = "maloney-geog-473"
GCS_BUCKET = "gis-final-project"
GCS_PREFIX_1307 = "GIS Final Project/1307"
# Optional extra prefixes to sync under /content.
# Keep this minimal for Colab disk limits: sync only BradySOM inputs, not full BradyGDB.
GCS_EXTRA_SYNC_PREFIXES = [
    "GIS Final Project/BradyGDB/BradyRaw/Brady_Analysis/Geophysics/BradySOM"
]

LOCAL_1307_DIR = "/content/1307"
REQUIREMENTS_FILE = "requirements.txt"  # relative to LOCAL_1307_DIR
EXTRA_PIP_PACKAGES = ""  # optional, space-separated

# Entry defaults; can be auto-overridden from README.md command examples.
# Default flow builds dataset first; switch to doe_geoai.py after dataset creation.
ENTRY_SCRIPT = "create_doe_dataset.py"
ENTRY_ARGS = ""  # leave blank to auto-build args for create_doe_dataset.py / doe_geoai.py from DOE_* values below
AUTO_CONFIG_FROM_README = False
README_REL_PATH = "README.md"  # relative to LOCAL_1307_DIR
PREFER_README_SCRIPT = "create_doe_dataset.py"  # choose this command when multiple are found

# Optional dataset-build step (README workflow) via create_doe_dataset.py.
# Set True only if a .gri exists under LOCAL_1307_DIR after GCS sync (or set DOE_GRI_INPUT).
# If you already have a built dataset, use False and set DOE_DATASET_PATH below.
DOE_BUILD_DATASET = False
DOE_GRI_INPUT = "/content/GIS Final Project/BradyGDB/BradyRaw/Brady_Analysis/Geophysics/BradySOM/brady_som_output.gri"  # stable default for Brady dataset build
DOE_DATASET_OUT_DIR = "/content/doe-data/brady_samples_19x3d"
DOE_CHANNELS = 3
DOE_SAMPLE_COUNT = 100000
DOE_KERNEL_PIXELS = 19

# doe_geoai.py required/optional inputs.
DOE_DATASET_PATH = ""   # required: -d / --dataset (directory)
DOE_LABELBIN_PATH = ""  # optional override for -l; defaults under LOCAL_RUN_DIR when blank

# Optional output overrides (defaults to LOCAL_RUN_DIR if blank).
DOE_MODEL_PATH = ""   # -m / --model
DOE_PLOT_PATH = ""    # -p / --plot
DOE_CURVES_PATH = ""  # -o / --output_curves

# Common training knobs for auto-built doe_geoai.py args.
DOE_EPOCHS = 25
DOE_BATCH_SIZE = 32
DOE_GPUS = 1
DOE_EXTRA_ARGS = ""  # optional extra flags, e.g. "-a -v"

# Pipeline helpers.
PIPELINE_RUN_BUILD = True
PIPELINE_RUN_TRAIN = True
PIPELINE_SYNC_AFTER_BUILD = True
PIPELINE_SYNC_AFTER_TRAIN = True

# Persistent run outputs in GCS.
GCS_OUTPUT_PREFIX = "GIS Final Project/outputs/1307"
RUN_NAME_OVERRIDE = ""  # leave blank for timestamped run names

# Optional dataset sync destination (recommended to avoid rebuilding every session).
SYNC_DATASET_TO_GCS = True
GCS_DATASET_PREFIX = "GIS Final Project/outputs/1307/datasets"

# Optional: auto-append output args for scripts that support these flags.
AUTO_APPEND_OUTPUT_ARGS = False
OUTPUT_DIR_FLAG = "--output_dir"
SAVE_DIR_FLAG = "--save_dir"

## 2) Authenticate and sync from GCS

In [ ]:
import shlex
import subprocess
from pathlib import Path

from google.colab import auth

def run(cmd, cwd=None):
    print("$", " ".join(shlex.quote(str(c)) for c in cmd))
    p = subprocess.run(
        cmd,
        cwd=cwd,
        text=True,
        capture_output=True,
    )
    if p.stdout:
        print(p.stdout)
    if p.returncode != 0:
        if p.stderr:
            print(p.stderr)
        raise subprocess.CalledProcessError(p.returncode, cmd, output=p.stdout, stderr=p.stderr)
    return p

if not GCP_PROJECT or not GCS_BUCKET:
    raise ValueError("Set GCP_PROJECT and GCS_BUCKET in the config cell first.")

auth.authenticate_user()
run(["gcloud", "config", "set", "project", GCP_PROJECT])

local_dir = Path(LOCAL_1307_DIR)
local_dir.mkdir(parents=True, exist_ok=True)

_prefix = GCS_PREFIX_1307.strip("/")
src = f"gs://{GCS_BUCKET}/{_prefix}" if _prefix else f"gs://{GCS_BUCKET}"
run(["gsutil", "-m", "rsync", "-r", src, str(local_dir)])
print("Synced 1307 code to", local_dir)

# Optional extra data syncs go under /content preserving relative path.
for extra_prefix in GCS_EXTRA_SYNC_PREFIXES:
    ep = extra_prefix.strip("/")
    if not ep:
        continue
    extra_src = f"gs://{GCS_BUCKET}/{ep}"
    extra_dst = Path("/content") / ep
    # gsutil rsync expects destination directory to exist on local filesystem.
    extra_dst.mkdir(parents=True, exist_ok=True)
    try:
        run(["gsutil", "-m", "rsync", "-r", extra_src, str(extra_dst)])
        print("Synced extra prefix via rsync:", extra_src, "->", extra_dst)
    except subprocess.CalledProcessError as exc:
        print("rsync failed for", extra_src)
        if exc.stderr:
            print(exc.stderr)
        print("Falling back to gsutil cp -r for this prefix...")
        try:
            run(["gsutil", "-m", "cp", "-r", extra_src, str(extra_dst.parent)])
            print("Synced extra prefix via cp -r:", extra_src, "->", extra_dst.parent)
        except subprocess.CalledProcessError as exc2:
            if exc2.stderr:
                print(exc2.stderr)
            raise RuntimeError(
                f"Failed syncing {extra_src}. "
                "Set GCS_EXTRA_SYNC_PREFIXES to exact small prefixes and rerun this cell."
            ) from exc2

$ gcloud config set project maloney-geog-473
$ gsutil -m rsync -r 'gs://gis-final-project/GIS Final Project/1307' /content/1307
Synced to /content/1307


## 3) Inspect synced files

In [ ]:
from pathlib import Path

root = Path(LOCAL_1307_DIR)
if not root.exists():
    raise FileNotFoundError(f"Missing local code directory: {root}")

items = sorted(p.name for p in root.iterdir())
print(f"Top-level files/folders in {root}:")
for name in items[:200]:
    print(" -", name)

Top-level files/folders in /content/1307:
 - README.md
 - create_doe_dataset.py
 - displacement_som_r_scripts
 - doe-ann
 - doe_ann_map.py
 - doe_geoai.py
 - doe_tiff
 - lst_extract.sh
 - lst_r_scripts
 - mineral_markers
 - sbatch_scripts


## 4) Install dependencies

In [ ]:
import sys
import shlex
from pathlib import Path

req_path = Path(LOCAL_1307_DIR) / REQUIREMENTS_FILE

run([sys.executable, "-m", "pip", "install", "-U", "pip"])
if req_path.exists():
    run([sys.executable, "-m", "pip", "install", "-r", str(req_path)])
else:
    print(f"No requirements file found at: {req_path}")

extra = EXTRA_PIP_PACKAGES.strip()
if extra:
    run([sys.executable, "-m", "pip", "install", *shlex.split(extra)])

$ /usr/bin/python3 -m pip install -U pip
No requirements file found at: /content/1307/requirements.txt


## 5) Check GPU runtime

In [ ]:
import subprocess

try:
    run(["nvidia-smi"])
except subprocess.CalledProcessError:
    print("nvidia-smi failed. In Colab: Runtime -> Change runtime type -> GPU.")

try:
    import torch
    print("torch version:", torch.__version__)
    print("cuda available:", torch.cuda.is_available())
except Exception as e:
    print("Torch check skipped:", e)

$ nvidia-smi


FileNotFoundError: [Errno 2] No such file or directory: 'nvidia-smi'

## 6) Create persistent run directories

This prepares a local run folder and a matching GCS destination so outputs can be synced off Colab runtime disk.

In [ ]:
from datetime import datetime
from pathlib import Path

run_name = RUN_NAME_OVERRIDE.strip() or datetime.now().strftime("run_%Y%m%d_%H%M%S")
LOCAL_RUN_DIR = Path(f"/content/1307_runs/{run_name}")
LOCAL_RUN_DIR.mkdir(parents=True, exist_ok=True)

_output_prefix = GCS_OUTPUT_PREFIX.strip("/")
if not _output_prefix:
    raise ValueError("Set GCS_OUTPUT_PREFIX in the config cell.")

GCS_RUN_URI = f"gs://{GCS_BUCKET}/{_output_prefix}/{run_name}"

print("Run name:", run_name)
print("Local run dir:", LOCAL_RUN_DIR)
print("GCS run uri:", GCS_RUN_URI)

## 7) (Optional) Build DOE dataset from GRI (README workflow)

Set `DOE_BUILD_DATASET = True` in the config cell when you want `create_doe_dataset.py` to run **before** training. You need a **`.gri` file** under `/content/1307` after GCS sync (upload it into your bucket’s 1307 prefix if missing).

Leave `DOE_GRI_INPUT` blank to **auto-pick** when there is **exactly one** `.gri` in the tree; otherwise set the full path (e.g. `/content/1307/…/file.gri`).

Default in the config is **`DOE_BUILD_DATASET = False`** if you already have a dataset: then set `DOE_DATASET_PATH` in config (or rely on auto-detection in this step).

This creates the dataset directory expected by `doe_geoai.py -d` and auto-sets `DOE_DATASET_PATH` to `DOE_DATASET_OUT_DIR` when the build runs.

In [ ]:
import shlex
from pathlib import Path

root = Path(LOCAL_1307_DIR)
if not root.exists():
    raise FileNotFoundError(f"Missing LOCAL_1307_DIR: {root}")

def find_matches(patterns):
    out = []
    for pat in patterns:
        out.extend(root.rglob(pat))
    files = sorted({str(p.resolve()) for p in out if p.is_file()})
    return files

if DOE_BUILD_DATASET:
    raw = DOE_GRI_INPUT.strip()
    placeholder = "<your_actual_gri_file>" in raw
    gri = None
    if raw and not placeholder:
        p = Path(raw)
        if p.is_file():
            gri = p
    if gri is None:
        gri_files = sorted(
            {p for p in root.rglob("*") if p.is_file() and p.suffix.lower() == ".gri"}
        )
        print("GRI search under", root, "->", len(gri_files), "file(s)")
        for p in gri_files[:30]:
            print(" -", p)
        if len(gri_files) == 1:
            gri = gri_files[0]
            print("Auto-selected GRI:", gri)
        elif len(gri_files) == 0:
            raise FileNotFoundError(
                "No .gri under LOCAL_1307_DIR after sync. Options: (1) Upload the .gri into your GCS 1307 prefix and re-run sync; "
                "(2) Set DOE_GRI_INPUT to the full path under /content/1307/...; or (3) Set DOE_BUILD_DATASET=False and set DOE_DATASET_PATH to an existing dataset directory."
            )
        else:
            raise FileNotFoundError(
                "Multiple .gri files found; set DOE_GRI_INPUT in the config cell to the one you want:\n"
                + "\n".join(f"  {p}" for p in gri_files)
            )

    out_dir = Path(DOE_DATASET_OUT_DIR).resolve()
    out_dir.parent.mkdir(parents=True, exist_ok=True)

    build_cmd = [
        "python",
        str(root / "create_doe_dataset.py"),
        "-i", str(gri),
        "-c", str(DOE_CHANNELS),
        "-d", str(out_dir),
        "-s", str(DOE_SAMPLE_COUNT),
        "-k", str(DOE_KERNEL_PIXELS),
    ]
    print("$", " ".join(shlex.quote(c) for c in build_cmd))
    run(build_cmd, cwd=str(root))

    DOE_DATASET_PATH = str(out_dir)
    print("Built dataset and set DOE_DATASET_PATH:", DOE_DATASET_PATH)
else:
    # Fallback auto-detection when dataset was prebuilt/uploaded.
    dataset_patterns = ["*samples*", "*dataset*", "*.h5", "*.hdf5", "*dataset*.npy", "*dataset*.npz"]
    labelbin_patterns = ["*label*bin*", "*label*.pickle", "*label*.pkl", "*label*.joblib", "*.l"]

    dataset_matches = find_matches(dataset_patterns)
    labelbin_matches = find_matches(labelbin_patterns)

    print("Dataset candidates:")
    for p in dataset_matches[:50]:
        print(" -", p)
    if not dataset_matches:
        print(" - none found")

    print("\nLabelbin candidates:")
    for p in labelbin_matches[:50]:
        print(" -", p)
    if not labelbin_matches:
        print(" - none found")

    if not DOE_DATASET_PATH.strip() and len(dataset_matches) == 1:
        DOE_DATASET_PATH = dataset_matches[0]
        print("\nAuto-set DOE_DATASET_PATH:", DOE_DATASET_PATH)
    elif not DOE_DATASET_PATH.strip() and len(dataset_matches) > 1:
        print("\nMultiple dataset candidates found; set DOE_DATASET_PATH manually in config.")

    if not DOE_LABELBIN_PATH.strip() and len(labelbin_matches) == 1:
        DOE_LABELBIN_PATH = labelbin_matches[0]
        print("Auto-set DOE_LABELBIN_PATH:", DOE_LABELBIN_PATH)
    elif not DOE_LABELBIN_PATH.strip() and len(labelbin_matches) > 1:
        print("Multiple labelbin candidates found; set DOE_LABELBIN_PATH manually in config.")

## 8) Write run metadata and execute your 1307 entry script

This stores the exact script/args/environment for reproducibility, then runs your job.

For `create_doe_dataset.py`, if `ENTRY_ARGS` is blank the notebook auto-builds args from `DOE_GRI_INPUT`, `DOE_DATASET_OUT_DIR`, and DOE sampling settings.
For `doe_geoai.py`, if `ENTRY_ARGS` is blank the notebook auto-builds required args from `DOE_DATASET_PATH` and output paths. Labels/model default under `LOCAL_RUN_DIR` when overrides are blank.

In [ ]:
import json
import platform
import re
import shlex
import sys
from datetime import datetime, timezone
from pathlib import Path

readme_path = Path(LOCAL_1307_DIR) / README_REL_PATH
args = ENTRY_ARGS.strip()

# Optionally derive entry command from README command examples in synced 1307 folder.
if AUTO_CONFIG_FROM_README and readme_path.is_file():
    text = readme_path.read_text(encoding="utf-8", errors="ignore")
    command_lines = []
    for line in text.splitlines():
        candidate = line.strip().strip("`")
        if re.match(r"^(python|python3)\s+\S+", candidate):
            command_lines.append(candidate)

    parsed = []
    for line in command_lines:
        try:
            tokens = shlex.split(line)
        except ValueError:
            continue
        if len(tokens) >= 2 and tokens[1].endswith(".py"):
            parsed.append(tokens)

    chosen = None
    preferred = PREFER_README_SCRIPT.strip()
    if preferred:
        for tokens in parsed:
            if Path(tokens[1]).name == preferred:
                chosen = tokens
                break
    if not chosen and parsed:
        chosen = parsed[0]

    if chosen:
        ENTRY_SCRIPT = Path(chosen[1]).name
        if not args and len(chosen) > 2:
            args = shlex.join(chosen[2:])
        print("README-derived command:", " ".join(chosen))
        print("Using ENTRY_SCRIPT:", ENTRY_SCRIPT)
        if args:
            print("Using args:", args)

entry = Path(LOCAL_1307_DIR) / ENTRY_SCRIPT
if not entry.exists():
    raise FileNotFoundError(
        f"ENTRY_SCRIPT not found: {entry}\n"
        "Update ENTRY_SCRIPT in the config cell."
    )

# Compatibility patch: synced 1307 tree has `doe_tiff/doe_tiff/*.py` but nested
# `__init__.py` used absolute imports (`from doe_tiff.io ...`) that break when only
# the nested package exists. Fix that, then import the nested package as `dt`.
root1307 = Path(LOCAL_1307_DIR)
nested_init = root1307 / "doe_tiff" / "doe_tiff" / "__init__.py"
if nested_init.is_file():
    init_txt = nested_init.read_text(encoding="utf-8", errors="ignore")
    init_patched = (
        init_txt.replace("from doe_tiff.io import", "from .io import")
        .replace("from doe_tiff.doe_kernel import", "from .doe_kernel import")
    )
    if init_patched != init_txt:
        nested_init.write_text(init_patched, encoding="utf-8")
        print("Patched doe_tiff/doe_tiff/__init__.py with relative imports")

if entry.name == "create_doe_dataset.py":
    src = entry.read_text(encoding="utf-8", errors="ignore")
    orig = src

    bad_block = (
        "try:\n"
        "    from doe_tiff.io import read_gdal_file, frame_image\n"
        "    from doe_tiff.doe_kernel import GeoTiffConvolution\n"
        "except Exception:\n"
        "    from doe_tiff.doe_tiff.io import read_gdal_file, frame_image\n"
        "    from doe_tiff.doe_tiff.doe_kernel import GeoTiffConvolution\n"
    )
    if bad_block in src:
        src = src.replace(bad_block, "")

    src = src.replace("import doe_tiff as dt", "import doe_tiff.doe_tiff as dt")
    src = src.replace("dt.io.read_gdal_file(", "dt.read_gdal_file(")

    # If earlier patches stripped `dt.` from helpers, restore it safely.
    src = re.sub(r"(?<!\.)read_gdal_file\(", "dt.read_gdal_file(", src)
    src = re.sub(r"(?<!\.)frame_image\(", "dt.frame_image(", src)
    src = re.sub(r"(?<!\.)GeoTiffConvolution\(", "dt.GeoTiffConvolution(", src)

    if src != orig:
        entry.write_text(src, encoding="utf-8")
        print("Patched create_doe_dataset.py for nested doe_tiff package layout")

if entry.name == "doe_geoai.py":
    src = entry.read_text(encoding="utf-8", errors="ignore")
    orig = src

    src = src.replace("import keras", "from tensorflow import keras")
    src = src.replace("from keras.callbacks import", "from tensorflow.keras.callbacks import")
    src = src.replace("from keras.layers import", "from tensorflow.keras.layers import")
    src = src.replace("from keras.regularizers import", "from tensorflow.keras.regularizers import")
    src = src.replace("from keras.layers.convolutional import", "from tensorflow.keras.layers import")
    src = src.replace("from keras.layers.core import", "from tensorflow.keras.layers import")
    src = src.replace("from keras.models import", "from tensorflow.keras.models import")
    src = src.replace("from keras.optimizers import", "from tensorflow.keras.optimizers import")
    src = src.replace("from keras.utils import", "from tensorflow.keras.utils import")
    src = src.replace("from keras import backend as K", "from tensorflow.keras import backend as K")

    src = src.replace(
        "from tensorflow.keras.utils import multi_gpu_model",
        "try:\n    from tensorflow.keras.utils import multi_gpu_model\nexcept Exception:\n    def multi_gpu_model(model, gpus=None):\n        return model"
    )

    if src != orig:
        entry.write_text(src, encoding="utf-8")
        print("Patched doe_geoai.py for modern tensorflow.keras imports")

cmd = [sys.executable, str(entry)]

# Auto-build args when ENTRY_ARGS is left blank.
if not args and entry.name == "create_doe_dataset.py":
    gri = DOE_GRI_INPUT.strip()
    if not gri:
        search_roots = [Path(LOCAL_1307_DIR), Path('/content')]
        found = []
        seen = set()
        for root in search_roots:
            if not root.exists():
                continue
            for p in root.rglob('*.gri'):
                rp = str(p.resolve())
                if rp not in seen:
                    seen.add(rp)
                    found.append(p.resolve())
        if len(found) == 1:
            gri = str(found[0])
            DOE_GRI_INPUT = gri
            print('Auto-detected DOE_GRI_INPUT:', gri)
        elif len(found) > 1:
            print('Found multiple .gri files; set DOE_GRI_INPUT explicitly:')
            for p in found[:25]:
                print(' -', p)
            raise ValueError(
                'create_doe_dataset.py needs -i/--image. '
                'Set DOE_GRI_INPUT in config to one of the files listed above.'
            )
        else:
            raise ValueError(
                'create_doe_dataset.py needs -i/--image. '
                'Set DOE_GRI_INPUT in config, or sync/copy a .gri into /content first.'
            )
    auto_args = [
        "-i", gri,
        "-c", str(DOE_CHANNELS),
        "-d", DOE_DATASET_OUT_DIR,
        "-s", str(DOE_SAMPLE_COUNT),
        "-k", str(DOE_KERNEL_PIXELS),
    ]
    cmd.extend(auto_args)
elif not args and entry.name == "doe_geoai.py":
    if not DOE_DATASET_PATH.strip():
        raise ValueError(
            "doe_geoai.py needs -d/--dataset. "
            "Set DOE_DATASET_PATH in config or run the dataset-build step."
        )

    dataset = DOE_DATASET_PATH.strip()
    labelbin = DOE_LABELBIN_PATH.strip() or str(Path(LOCAL_RUN_DIR) / "doe_labels.l")
    model = DOE_MODEL_PATH.strip() or str(Path(LOCAL_RUN_DIR) / "doe_geoai_model.h5")
    plot = DOE_PLOT_PATH.strip() or str(Path(LOCAL_RUN_DIR) / "doe_geoai_training_plot.png")
    curves = DOE_CURVES_PATH.strip() or str(Path(LOCAL_RUN_DIR) / "doe_geoai_training_curves.csv")

    auto_args = [
        "-d", dataset,
        "-l", labelbin,
        "-m", model,
        "-p", plot,
        "-o", curves,
        "-e", str(DOE_EPOCHS),
        "-b", str(DOE_BATCH_SIZE),
        "-g", str(DOE_GPUS),
        "-k", str(DOE_KERNEL_PIXELS),
        "-c", str(DOE_CHANNELS),
    ]
    extra = DOE_EXTRA_ARGS.strip()
    if extra:
        auto_args.extend(shlex.split(extra))

    cmd.extend(auto_args)
else:
    if args:
        cmd.extend(shlex.split(args))

if AUTO_APPEND_OUTPUT_ARGS:
    cmd.extend([
        OUTPUT_DIR_FLAG,
        str(LOCAL_RUN_DIR),
        SAVE_DIR_FLAG,
        str(LOCAL_RUN_DIR / "checkpoints"),
    ])

# Write a reproducible run manifest before execution.
manifest = {
    "created_utc": datetime.now(timezone.utc).isoformat(),
    "entry_script": ENTRY_SCRIPT,
    "entry_args": args,
    "command": cmd,
    "python_executable": sys.executable,
    "python_version": sys.version,
    "platform": platform.platform(),
    "gcp_project": GCP_PROJECT,
    "gcs_bucket": GCS_BUCKET,
    "gcs_code_prefix_1307": GCS_PREFIX_1307,
    "gcs_output_prefix": GCS_OUTPUT_PREFIX,
    "gcs_run_uri": GCS_RUN_URI,
    "local_code_dir": str(LOCAL_1307_DIR),
    "local_run_dir": str(LOCAL_RUN_DIR),
    "auto_append_output_args": AUTO_APPEND_OUTPUT_ARGS,
    "output_dir_flag": OUTPUT_DIR_FLAG,
    "save_dir_flag": SAVE_DIR_FLAG,
    "doe_build_dataset": DOE_BUILD_DATASET,
    "doe_gri_input": DOE_GRI_INPUT,
    "doe_dataset_out_dir": DOE_DATASET_OUT_DIR,
    "doe_channels": DOE_CHANNELS,
    "doe_sample_count": DOE_SAMPLE_COUNT,
    "doe_kernel_pixels": DOE_KERNEL_PIXELS,
    "doe_dataset_path": DOE_DATASET_PATH,
    "doe_labelbin_path": DOE_LABELBIN_PATH,
    "doe_model_path": DOE_MODEL_PATH,
    "doe_plot_path": DOE_PLOT_PATH,
    "doe_curves_path": DOE_CURVES_PATH,
    "doe_epochs": DOE_EPOCHS,
    "doe_batch_size": DOE_BATCH_SIZE,
    "doe_gpus": DOE_GPUS,
    "doe_extra_args": DOE_EXTRA_ARGS,
}
manifest_path = Path(LOCAL_RUN_DIR) / "run_config.json"
manifest_path.write_text(json.dumps(manifest, indent=2), encoding="utf-8")
print("Wrote", manifest_path)

run(cmd, cwd=LOCAL_1307_DIR)

## 9) One-click pipeline run (optional)

Runs successive steps in order using current config values:
1. Build dataset (`create_doe_dataset.py`) when `PIPELINE_RUN_BUILD=True`
2. Train model (`doe_geoai.py`) when `PIPELINE_RUN_TRAIN=True`
3. Sync dataset/run outputs to GCS based on `PIPELINE_SYNC_AFTER_*` + `SYNC_DATASET_TO_GCS`

Use this when you want a single execution flow instead of manually running section 7 then section 8 then section 10.

In [ ]:
import re
import shlex
import sys
from pathlib import Path

root1307 = Path(LOCAL_1307_DIR)
if not root1307.exists():
    raise FileNotFoundError(f"Missing LOCAL_1307_DIR: {root1307}")

if "LOCAL_RUN_DIR" not in globals() or "GCS_RUN_URI" not in globals():
    raise RuntimeError("Run section 6 first to initialize LOCAL_RUN_DIR and GCS_RUN_URI")

def _sync_path(local_path: Path, gcs_uri: str, label: str):
    if not local_path.exists():
        print(f"Skip sync ({label}): local path missing -> {local_path}")
        return
    run(["gsutil", "-m", "rsync", "-r", str(local_path), gcs_uri])
    print(f"Synced {label}: {local_path} -> {gcs_uri}")

def _patch_doe_geoai(path: Path):
    if not path.is_file():
        raise FileNotFoundError(f"doe_geoai.py not found: {path}")

    txt = path.read_text(encoding="utf-8", errors="ignore")
    orig = txt

    txt = txt.replace("import keras", "from tensorflow import keras")
    txt = txt.replace("from keras.callbacks import", "from tensorflow.keras.callbacks import")
    txt = txt.replace("from keras.layers import", "from tensorflow.keras.layers import")
    txt = txt.replace("from keras.regularizers import", "from tensorflow.keras.regularizers import")
    txt = txt.replace("from keras.layers.convolutional import", "from tensorflow.keras.layers import")
    txt = txt.replace("from keras.layers.core import", "from tensorflow.keras.layers import")
    txt = txt.replace("from keras.models import", "from tensorflow.keras.models import")
    txt = txt.replace("from keras.optimizers import", "from tensorflow.keras.optimizers import")
    txt = txt.replace("from keras.utils import", "from tensorflow.keras.utils import")
    txt = txt.replace("from keras import backend as K", "from tensorflow.keras import backend as K")

    txt = txt.replace(
        "from tensorflow.keras.utils import multi_gpu_model",
        "try:\n    from tensorflow.keras.utils import multi_gpu_model\nexcept Exception:\n    def multi_gpu_model(model, gpus=None):\n        return model"
    )

    if txt != orig:
        path.write_text(txt, encoding="utf-8")
        print("Patched doe_geoai.py for modern tensorflow.keras imports")

# Step A: dataset build
if PIPELINE_RUN_BUILD:
    if not DOE_GRI_INPUT.strip():
        raise ValueError("PIPELINE_RUN_BUILD=True requires DOE_GRI_INPUT")

    dataset_out = Path(DOE_DATASET_OUT_DIR).resolve()
    dataset_out.parent.mkdir(parents=True, exist_ok=True)

    build_cmd = [
        sys.executable,
        str(root1307 / "create_doe_dataset.py"),
        "-i", DOE_GRI_INPUT.strip(),
        "-c", str(DOE_CHANNELS),
        "-d", str(dataset_out),
        "-s", str(DOE_SAMPLE_COUNT),
        "-k", str(DOE_KERNEL_PIXELS),
    ]
    print("$", " ".join(shlex.quote(c) for c in build_cmd))
    run(build_cmd, cwd=str(root1307))

    DOE_DATASET_PATH = str(dataset_out)
    print("PIPELINE: dataset ready ->", DOE_DATASET_PATH)

    if PIPELINE_SYNC_AFTER_BUILD and SYNC_DATASET_TO_GCS:
        ds_prefix = GCS_DATASET_PREFIX.strip("/")
        ds_uri = f"gs://{GCS_BUCKET}/{ds_prefix}/{Path(DOE_DATASET_PATH).name}"
        _sync_path(Path(DOE_DATASET_PATH), ds_uri, "dataset")
else:
    print("PIPELINE: dataset build skipped (PIPELINE_RUN_BUILD=False)")

# Step B: training
if PIPELINE_RUN_TRAIN:
    if not DOE_DATASET_PATH.strip():
        raise ValueError("PIPELINE_RUN_TRAIN=True requires DOE_DATASET_PATH")

    doe_geoai = root1307 / "doe_geoai.py"
    _patch_doe_geoai(doe_geoai)

    labelbin = DOE_LABELBIN_PATH.strip() or str(Path(LOCAL_RUN_DIR) / "doe_labels.l")
    model = DOE_MODEL_PATH.strip() or str(Path(LOCAL_RUN_DIR) / "doe_geoai_model.h5")
    plot = DOE_PLOT_PATH.strip() or str(Path(LOCAL_RUN_DIR) / "doe_geoai_training_plot.png")
    curves = DOE_CURVES_PATH.strip() or str(Path(LOCAL_RUN_DIR) / "doe_geoai_training_curves.csv")

    train_cmd = [
        sys.executable,
        str(doe_geoai),
        "-d", DOE_DATASET_PATH.strip(),
        "-l", labelbin,
        "-m", model,
        "-p", plot,
        "-o", curves,
        "-e", str(DOE_EPOCHS),
        "-b", str(DOE_BATCH_SIZE),
        "-g", str(DOE_GPUS),
        "-k", str(DOE_KERNEL_PIXELS),
        "-c", str(DOE_CHANNELS),
    ]
    extra = DOE_EXTRA_ARGS.strip()
    if extra:
        train_cmd.extend(shlex.split(extra))

    print("$", " ".join(shlex.quote(c) for c in train_cmd))
    run(train_cmd, cwd=str(root1307))

    if PIPELINE_SYNC_AFTER_TRAIN:
        _sync_path(Path(LOCAL_RUN_DIR), GCS_RUN_URI, "run outputs")
else:
    print("PIPELINE: training skipped (PIPELINE_RUN_TRAIN=False)")

## 10) Sync outputs to GCS

Run this during training and after training to persist checkpoints/logs/results.

If step 8 stops with missing DOE dataset path, set `DOE_BUILD_DATASET=True` and `DOE_GRI_INPUT` in config, then run step 7 to build it.

In [ ]:
from pathlib import Path

if "LOCAL_RUN_DIR" not in globals() or "GCS_RUN_URI" not in globals():
    raise RuntimeError("Run the persistent run directory cell first.")

if not Path(LOCAL_RUN_DIR).exists():
    raise FileNotFoundError(f"Missing local run directory: {LOCAL_RUN_DIR}")

run(["gsutil", "-m", "rsync", "-r", str(LOCAL_RUN_DIR), GCS_RUN_URI])
print("Synced:", GCS_RUN_URI)